<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/04_agentic_ai.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Building an AI Course Agent with the OpenAI Agents SDK</h2>
            <h4 style="color:#00bf00;">Copyright 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">In this lab, we turn the course assistant from Lab 3 into an agent. The OpenAI Agents SDK manages the agent loop and tool calls, while GPT-OSS:120B runs locally through the Ollama OpenAI-compatible API. We will add Phoenix tracing, an email-recording tool, and a Gradio chat interface.</span>
        </td>
    </tr>
</table>

Run the notebook from top to bottom. Set `OLLAMA_BASE_URL` and `PHOENIX_API_KEY` in the project-root `.env` file. Phoenix runs on port `6006` on the same server as Ollama.

## What you will build

- An `Agent` configured with course-specific instructions.
- A Python function exposed to the model as a tool.
- An agent loop managed by the OpenAI Agents SDK (instead of hand-written tool-call handling).
- A Gradio chat interface that keeps conversation history.

The model runs on your configured Ollama server. OpenAI Agents SDK traces are exported to the Phoenix server; the API key is read from the project-root `.env` and is never printed.

In [1]:
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import gradio as gr
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI
from pypdf import PdfReader
from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from phoenix.otel import register

## 1. Connect to the local model

Ollama provides an OpenAI-compatible Chat Completions endpoint. We use the Agents SDK's `OpenAIChatCompletionsModel` adapter so the SDK sends requests to that local endpoint. The API key is a placeholder required by the client; Ollama does not use it for local authentication.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
load_dotenv(PROJECT_ROOT / ".env", override=True)

MODEL = "gpt-oss:120b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

PHOENIX_API_KEY = os.getenv("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    raise ValueError("Set PHOENIX_API_KEY in the project-root .env file")

# Phoenix is on the same host as Ollama, using port 6006.
ollama_url = urlparse(OLLAMA_BASE_URL)
phoenix_host = ollama_url.hostname
if not phoenix_host:
    raise ValueError("OLLAMA_BASE_URL must include a server hostname")
if ":" in phoenix_host:
    phoenix_host = f"[{phoenix_host}]"
PHOENIX_ENDPOINT = f"{ollama_url.scheme}://{phoenix_host}:6006/v1/traces"

tracer_provider = register(
    project_name="023605248", #<----------student id
    endpoint=PHOENIX_ENDPOINT,
    api_key=PHOENIX_API_KEY,
)
# To use Phoenix with Ollama, go to http://SERVER_IP:6006 login with user name: student@braude.ac.il and password:1234567890 add new project with your id and use it as project name
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

ollama_client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)
agent_model = OpenAIChatCompletionsModel(
    model=MODEL,
    openai_client=ollama_client,
)
print(f"Configured local model: {MODEL}")
print(f"Ollama endpoint: {OLLAMA_BASE_URL}")
print(f"Phoenix project: lab ({PHOENIX_ENDPOINT})")

OpenTelemetry Tracing Details
|  Phoenix Project: 023605248
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://100.126.157.125:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Configured local model: gpt-oss:120b
Ollama endpoint: http://100.126.157.125:11434/v1
Phoenix project: lab (http://100.126.157.125:6006/v1/traces)


In [3]:
# Confirm the model is available before creating the agent.
model_check_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
available_models = [model.id for model in model_check_client.models.list().data]
print("Available Ollama models:", available_models)
if MODEL not in available_models:
    raise ValueError(f"{MODEL!r} is not available on the configured Ollama server")

Available Ollama models: ['gpt-oss:20b', 'gpt-oss:20b', 'llamacpp:9ba9cc2b4e02463b933096090530aef679d534dd91ecc9afadfb2dbfdc59f3c5', 'nemotron-3-super:120b-a12b-q4_K_M', 'qwen3.5:122b', 'qwen3.5:122b', 'llamacpp:b48ba378786285c5bf74134fda36f02dcc0025ac52d1aafe0e4fc2ce87a3b391', 'qwen3.8:27b', 'gpt-oss:120b', 'gpt-oss:120b', 'llamacpp:ad84bf7720de3aac13b8f07008047c85ff5c277721a648dbf51517453a0331f6', 'tev1:0.8b', 'tev1:latest', 'nimble:latest', 'gemma3:27b', 'qwen3:30b', 'qwen2.5:latest', 'qwen3.8:latest', 'llama3.1:8b']


## 2. Load the course material

These files are included in the Lab 4 folder so the notebook can run independently of Lab 3. This cell works whether Jupyter starts in the lab folder or the project root.

In [4]:
LAB_DIR = PROJECT_ROOT / "labs" / "4_lab4"

SUMMARY_PATH = LAB_DIR / "docs" / "summary.txt"
SYLLABUS_PATH = LAB_DIR / "docs" / "syllabus.pdf"
if not SUMMARY_PATH.exists() or not SYLLABUS_PATH.exists():
    raise FileNotFoundError("Course documents not found. Run this notebook from the project or Lab 4 folder.")

summary = SUMMARY_PATH.read_text(encoding="utf-8")
reader = PdfReader(SYLLABUS_PATH)
syllabus = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Loaded course summary ({len(summary)} characters)")
print(f"Loaded syllabus ({len(syllabus)} characters)")

Loaded course summary (410 characters)
Loaded syllabus (2420 characters)


## 3. Define the agent's instructions

Instructions describe the agent's role and boundaries. They are different from tools: instructions guide behavior, while tools let the agent perform an action.

In [5]:
instructions = f"""
You are the digital assistant for the AI Agents Course. Be professional, friendly,
and helpful to people interested in the course. Answer questions about the course,
its content, and the information in the supplied materials.

COURSE SUMMARY
{summary}

SYLLABUS
{syllabus}

Rules:
- If the answer is not supported by these materials, say that you do not know.
- Stay focused on the course and its professional topics.
- You are an AI assistant; do not claim to be a human.
- When the user shares an email address, you MUST call the record_email tool
  before responding, unless the user explicitly says not to save it. Use the
  tool result to confirm success; never claim success without it.
- The user may share the address in Hebrew, for example: "זה המייל שלי: name@example.com".
- Tell the user that the address is stored locally in Lab 4.
"""

## 4. Give the agent a tool

The `@function_tool` decorator turns a typed Python function into a tool the agent can select. The function writes only to the local `emails.txt` file in the Lab 4 folder.

In [6]:
EMAILS_FILE = LAB_DIR / "emails.txt"

EMAIL_PATTERN = re.compile(r"[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+")

def email_is_saved(email: str) -> bool:
    if not EMAILS_FILE.exists():
        return False
    return any(line.strip().casefold() == email.casefold() for line in EMAILS_FILE.read_text(encoding="utf-8").splitlines())

def save_email_locally(email: str) -> str:
    email = email.strip().rstrip(".,;:!?")
    if "\n" in email or "\r" in email or not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
        return "That does not look like a valid email address, so it was not saved."
    if email_is_saved(email):
        return f"Email is already saved locally at {EMAILS_FILE.resolve()}."

    with EMAILS_FILE.open("a", encoding="utf-8") as file:
        file.write(email + "\n")
    print(f"Email saved locally at {EMAILS_FILE.resolve()}")
    return f"Email saved locally at {EMAILS_FILE.resolve()}."

@function_tool
def record_email(email: str) -> str:
    """Save an email address shared by a course visitor in Lab 4 emails.txt, unless they ask not to."""
    return save_email_locally(email)

print("Email tool is ready. It will write to:", EMAILS_FILE)

Email tool is ready. It will write to: c:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt


The SDK reads the function name, docstring, and type annotations to describe the tool to the model. Unlike Lab 3, we do not manually write a JSON schema, inspect `finish_reason`, or submit tool results ourselves.

In [7]:
agent = Agent(
    name="AI Course Assistant",
    instructions=instructions,
    model=agent_model,
    tools=[record_email],
)

print(agent.name)

AI Course Assistant


## 5. Run the agent

In a notebook, an event loop is already running, so use the asynchronous `Runner.run` and top-level `await`. The SDK decides whether to answer directly or call a tool, runs the tool, passes its result back to the model, and returns the final response.

In [8]:
result = await Runner.run(agent, "Hello! What is this course about?")
print(result.final_output)

The AI Agents & Agentic Workflows course, taught by Dr. Elena Karamer at Braude College, gives you a solid foundation in building **autonomous AI agents** and the workflows that power them.

**What you’ll learn**

| Topic | What it means for you |
|------|-----------------------|
| **LLMs & Agentic AI Development** | How large‑language models become decision‑making “engines.” |
| **OpenAI SDK & Structured Outputs** | Using the SDK to get reliable, parsable results from LLMs. |
| **Model Context Protocol (MCP)** | A communication protocol that lets agents share context efficiently. |
| **High‑Performance Prompt Engineering** | Designing prompts that make agents act predictably and efficiently. |
| **Retrieval‑Augmented Generation (RAG)** | Adding external knowledge bases so agents can fetch up‑to‑date information. |
| **From Workflows to Autonomous Agents** | Turning linear pipelines into self‑directed agents. |
| **Design Patterns for Agents & Workflows** | Re‑using proven architectura

Try sharing an email address in the chat. It will be saved to the Lab 4 `emails.txt` file unless you explicitly ask the assistant not to save it. The optional sample below is blank, so Run All will not append a placeholder.

In [9]:
test_email = ""  # Enter an address only if you want to save it as a test.
if test_email.strip():
    result = await Runner.run(
        agent,
        f"Please save my email {test_email.strip()} so the course team can contact me.",
    )
    print(result.final_output)
else:
    print("Email test skipped; no placeholder address will be saved.")

Email test skipped; no placeholder address will be saved.


## 6. Add a chat interface

Gradio passes the conversation history to `chat`. We send that history and the latest message to the SDK; the `Agent` keeps its instructions and tools for every turn.

In [10]:
NEGATIVE_SAVE_REQUEST = re.compile(
    r"\b(?:don't|do not|never)\s+(?:(?:want|need)\s+you\s+to\s+)?(?:save|store|record)\b"
    r"|אל\s+ת(?:שמור|תעד|תרשום)|לא\s+(?:(?:רוצה|צריך)\s+)?(?:לשמור|לתעד|לרשום)",
    re.IGNORECASE,
)

async def chat(message, history):
    conversation = []
    for turn in history:
        role = turn.get("role")
        content = turn.get("content")
        if role in {"user", "assistant"} and isinstance(content, str):
            conversation.append({"role": role, "content": content})

    email_match = EMAIL_PATTERN.search(message)
    email_to_record = None
    was_saved_before = False
    if email_match and not NEGATIVE_SAVE_REQUEST.search(message):
        email_to_record = email_match.group(0)
        was_saved_before = email_is_saved(email_to_record)

    conversation.append({"role": "user", "content": message})
    result = await Runner.run(agent, input=conversation)

    # The model gets first chance to use the tool. This fallback guarantees that
    # a shared address is recorded if a local model skips the tool call.
    if email_to_record:
        if was_saved_before:
            return f"Email was already saved locally at {EMAILS_FILE.resolve()}."
        if email_is_saved(email_to_record):
            return f"Email saved locally at {EMAILS_FILE.resolve()}."
        return save_email_locally(email_to_record)

    return result.final_output

await chat("Please summarize the course in one sentence.", [])

'The AI Agents & Agentic Workflows course teaches students to design, develop, and secure autonomous, multi‑agent AI systems—leveraging LLM inference, Retrieval‑Augmented Generation, advanced protocols, and design patterns—to create end‑to‑end intelligent solutions.'

In [11]:
gr.ChatInterface(
    fn=chat,
    title="AI Course Assistant",
    description="Ask about the course. Email addresses you share are saved locally in Lab 4 unless you ask not to.",
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Email saved locally at C:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt
Email saved locally at C:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt
Email saved locally at C:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt
Email saved locally at C:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt
Email saved locally at C:\Users\Levi\Desktop\Ai Course\Agents Course\labs\4_lab4\emails.txt


# Congratulations!

You built a course assistant with the OpenAI Agents SDK, connected it to a local GPT-OSS:120B model, gave it a tool, and launched a chat interface.

Compared with Lab 3, the SDK now manages the agent loop and tool execution. The Python function remains your implementation; the SDK makes it available to the agent and handles the back-and-forth with the model.